## Create the ML model!

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report

### Data 

In [ ]:
npz = np.load('Audiobooks_train.npz')
train_inputs = npz['inputs'].astype(np.float64)
train_targets = npz['targets'].astype(np.int64)

npz = np.load('Audiobooks_validation.npz')
val_inputs = npz['inputs'].astype(np.float64)
val_targets = npz['targets'].astype(np.int64)

npz = np.load('Audiobooks_test.npz')
test_inputs = npz['inputs'].astype(np.float64)
test_targets = npz['targets'].astype(np.int64)

### Model

##### Optimizers, losses and metrics

In [ ]:
input_size = 10
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='softmax'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

BATCH_SIZE = 50
EPOCHS = 100

stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2)

history = model.fit(
    train_inputs, train_targets,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[stopping],
    validation_data=(val_inputs, val_targets),
    verbose=2
)

### Training curves — Accuracy & Loss

In [ ]:
epochs_ran = range(1, len(history.history['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history.history['accuracy'],     label='Train',      color='#2196F3', linewidth=2)
ax1.plot(epochs_ran, history.history['val_accuracy'], label='Validation', color='#FF9800', linewidth=2, linestyle='--')
ax1.set_title('Model Accuracy per Epoch', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, history.history['loss'],     label='Train',      color='#2196F3', linewidth=2)
ax2.plot(epochs_ran, history.history['val_loss'], label='Validation', color='#FF9800', linewidth=2, linestyle='--')
ax2.set_title('Model Loss per Epoch', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Testing the model

In [ ]:
test_loss, test_acc = model.evaluate(test_inputs, test_targets, verbose=2)
print(f'Test accuracy: {test_acc*100:.2f}%')

### Confusion Matrix

In [ ]:
pred_probs = model.predict(test_inputs)
pred_classes = np.argmax(pred_probs, axis=1)

cm = confusion_matrix(test_targets, pred_classes)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d',
    cmap='Blues',
    xticklabels=['Predicted: No', 'Predicted: Yes'],
    yticklabels=['Actual: No',    'Actual: Yes'],
    linewidths=0.5,
    ax=ax
)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### Classification Report

In [ ]:
print(classification_report(
    test_targets, pred_classes,
    target_names=['Will not return', 'Will return']
))